In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/viki2310/dataset-2/retailer_week_ml_dataset_final.csv


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Update the path to point to your dataset on Kaggle
# Example: /kaggle/input/your-dataset-name/retailer_week_ml_dataset_improved.csv
DATA_PATH = "/kaggle/input/datasets/viki2310/dataset-2/retailer_week_ml_dataset_final.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")


Dataset shape: (120000, 35)

First 5 rows:


In [3]:
df.head().T

,0,1,2,3,4
retailer_id,RTL_00001,RTL_00001,RTL_00001,RTL_00001,RTL_00001
territory_id,TER_0001,TER_0001,TER_0001,TER_0001,TER_0001
state,Bihar,Bihar,Bihar,Bihar,Bihar
district,Patna,Patna,Patna,Patna,Patna
tehsil,Patna_T012,Patna_T012,Patna_T012,Patna_T012,Patna_T012
week,2025-10-06,2025-10-13,2025-10-20,2025-10-27,2025-11-03
total_inventory,328.0,322.0,297.0,287.0,287.0
out_of_stock_skus,0.0,0.0,0.0,0.0,0.0
unique_skus,3.0,3.0,3.0,3.0,3.0
avg_inventory,109.333333,107.333333,99.0,95.666667,95.666667


In [51]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Percentage", ascending=False)

if len(missing_df) == 0:
    print("✅ No missing values found.")
else:
    print("⚠️ Missing values detected:")
    display(missing_df)

✅ No missing values found.


In [52]:
# ... after splitting data

# 1. Identify categorical columns from the full dataset (before dropping)
cat_cols_full = df.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Create X_train and X_test
X_train = train_df.drop(columns=['target', 'week', 'retailer_id', 'territory_id'])
y_train = train_df['target']
X_test = test_df.drop(columns=['target', 'week', 'retailer_id', 'territory_id'])
y_test = test_df['target']

# 3. Filter categorical columns to only those that are still in X_train
cat_cols_train = [col for col in cat_cols_full if col in X_train.columns]
print("Categorical columns used in training:", cat_cols_train)

# 4. Train CatBoost with filtered categorical columns
cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=7,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    task_type='GPU',
    verbose=100,
    cat_features=cat_cols_train   # <-- Only columns present in X_train
)

cat_model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

Categorical columns used in training: ['state', 'district', 'tehsil']


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7850701	best: 0.7850701 (0)	total: 44.7ms	remaining: 1m 29s
100:	test: 0.7895383	best: 0.7899361 (65)	total: 2.61s	remaining: 49.1s
200:	test: 0.7894321	best: 0.7899361 (65)	total: 5.04s	remaining: 45.1s
300:	test: 0.7855060	best: 0.7899361 (65)	total: 7.47s	remaining: 42.2s
400:	test: 0.7823260	best: 0.7899361 (65)	total: 9.98s	remaining: 39.8s
500:	test: 0.7788822	best: 0.7899361 (65)	total: 12.4s	remaining: 37.1s
600:	test: 0.7749365	best: 0.7899361 (65)	total: 14.8s	remaining: 34.4s
700:	test: 0.7733825	best: 0.7899361 (65)	total: 17.3s	remaining: 32s
800:	test: 0.7703085	best: 0.7899361 (65)	total: 19.9s	remaining: 29.8s
900:	test: 0.7684218	best: 0.7899361 (65)	total: 22.6s	remaining: 27.6s
1000:	test: 0.7634541	best: 0.7899361 (65)	total: 25.2s	remaining: 25.2s
1100:	test: 0.7626224	best: 0.7899361 (65)	total: 27.8s	remaining: 22.7s
1200:	test: 0.7608618	best: 0.7899361 (65)	total: 30.4s	remaining: 20.2s
1300:	test: 0.7576834	best: 0.7899361 (65)	total: 32.9s	remainin

CatBoostClassifier(auto_class_weights='Balanced', cat_features=['state', 'district', 'tehsil'], depth=7, eval_metric='AUC', iterations=2000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [53]:
y_prob = cat_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_threshold = 0.5

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best Threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

Best Threshold: 0.60, F1: 0.5648


In [54]:
y_pred_final = (y_prob >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_final))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.89      0.72      0.79     39201
           1       0.46      0.74      0.56     12799

    accuracy                           0.72     52000
   macro avg       0.68      0.73      0.68     52000
weighted avg       0.79      0.72      0.74     52000

ROC AUC: 0.7899360712336908


In [55]:
# Convert all object columns to category
for col in X_train.select_dtypes(include=['object']).columns:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# Now LightGBM will accept them
cat_cols = X_train.select_dtypes(include=['category']).columns.tolist()
print("Categorical columns (category dtype):", cat_cols)

Categorical columns (category dtype): ['state', 'district', 'tehsil']


In [56]:
for col in X_train.select_dtypes(include=['object']).columns:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

cat_cols = X_train.select_dtypes(include=['category']).columns.tolist()

# 2. Train LightGBM on CPU
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    device='cpu',            # <-- Use CPU to avoid GPU bin limit
    verbose=50,
    categorical_feature=cat_cols
)

lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], 
              eval_metric='auc', callbacks=[lgb.early_stopping(50)])

# 3. Evaluate
y_prob = lgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)  # You can tune threshold later

print(classification_report(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] categorical_feature is set=state,district,tehsil, categorical_column=0,1,2 will be ignored. Current value: categorical_feature=state,district,tehsil
[LightGBM] [Info] Number of positive: 12418, number of negative: 55582
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.796335
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.381772
[LightGBM] [Debug] init for col-wise cost 0.006971 seconds, init for row-wise cost 0.019295 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023534 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5557
[LightGBM] [Info] Number of data points in the train set: 68000, number of 

In [62]:
# After training both models
y_prob_cat = cat_model.predict_proba(X_test)[:, 1]
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

y_prob_ensemble = (y_prob_cat + y_prob_lgb) / 2

# Find best threshold for ensemble
best_f1_ensemble = 0
best_threshold_ensemble = 0.5
for t in thresholds:
    y_pred = (y_prob_ensemble >= t).astype(int)
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1_ensemble:
        best_f1_ensemble = f1
        best_threshold_ensemble = t

print(f"Ensemble F1: {best_f1_ensemble:.4f}")

Ensemble F1: 0.5648
